# 01 – QA kiểm tra dữ liệu gốc (BRFSS 2015)

Chạy từ thư mục `notebooks/` hoặc thư mục gốc repo. Mọi bước dùng `assert`: nếu số liệu lệch quy chuẩn, notebook dừng và báo lỗi.

**Quy tắc dữ liệu của nhóm:** giữ nguyên toàn bộ 253.680 bản ghi, không dùng `drop_duplicates()`, không sửa nhãn.

## 1. Đọc dữ liệu và tính SHA-256

In [2]:
import hashlib
import json
from pathlib import Path
import pandas as pd

FILE_NAME = 'diabetes_binary_health_indicators_BRFSS2015.csv'
TARGET = 'Diabetes_binary'

# Tìm file dữ liệu dù chạy từ notebooks/ hay từ thư mục gốc
candidates = [Path('../data') / FILE_NAME, Path('data') / FILE_NAME]
file_path = next((p for p in candidates if p.is_file()), None)
assert file_path is not None, f'Không tìm thấy {FILE_NAME}. Tải từ Kaggle và đặt vào data/ (xem README).'

sha256 = hashlib.sha256(file_path.read_bytes()).hexdigest()
df = pd.read_csv(file_path)
print('File     :', file_path.resolve())
print('SHA-256  :', sha256)
print('Kích thước:', df.shape)

File     : C:\Users\User Guest\.vscode\UET.CS2046_7_Nhom5-1\data\diabetes_binary_health_indicators_BRFSS2015.csv
SHA-256  : 19f367e3e3350768f0c144c5d73ee5b355f67a57eaaa86ca7bd8aec594d8b1d0
Kích thước: (253680, 22)


## 2. Kiểm tra quy chuẩn (dừng nếu lệch)

In [3]:
EXPECTED = dict(rows=253_680, cols=22, positives=35_346, duplicates=24_206)

assert df.shape == (EXPECTED['rows'], EXPECTED['cols']), f'Sai kích thước: {df.shape}'
assert TARGET in df.columns, f'Thiếu cột nhãn {TARGET}'
assert set(df[TARGET].unique()) == {0, 1}, 'Nhãn phải chỉ gồm 0 và 1'
assert int(df[TARGET].sum()) == EXPECTED['positives'], 'Sai số mẫu dương'
assert df.isna().sum().sum() == 0, 'Dữ liệu có giá trị thiếu'

n_dup = int(df.duplicated().sum())
assert n_dup == EXPECTED['duplicates'], f'Sai số dòng trùng: {n_dup}'

prevalence = df[TARGET].mean()
print(f'Tỷ lệ dương: {prevalence:.4%} ({int(df[TARGET].sum()):,} / {len(df):,})')
print(f'Dòng trùng hoàn toàn: {n_dup:,}  -> GIỮ NGUYÊN, không dùng drop_duplicates()')

Tỷ lệ dương: 13.9333% (35,346 / 253,680)
Dòng trùng hoàn toàn: 24,206  -> GIỮ NGUYÊN, không dùng drop_duplicates()


## 3. Kiểm tra miền giá trị

In [4]:
feat = df.drop(columns=TARGET)

BINARY = ['HighBP','HighChol','CholCheck','Smoker','Stroke','HeartDiseaseorAttack',
          'PhysActivity','Fruits','Veggies','HvyAlcoholConsump','AnyHealthcare',
          'NoDocbcCost','DiffWalk','Sex']
RANGES = {'GenHlth': (1, 5), 'Age': (1, 13), 'Education': (1, 6), 'Income': (1, 8),
          'MentHlth': (0, 30), 'PhysHlth': (0, 30)}

assert feat.shape[1] == 21, f'Phải có đúng 21 đặc trưng, hiện có {feat.shape[1]}'
for c in BINARY:
    assert feat[c].isin([0, 1]).all(), f'{c} có giá trị ngoài 0/1'
for c, (lo, hi) in RANGES.items():
    assert feat[c].between(lo, hi).all(), f'{c} ngoài miền {lo}..{hi}'
    assert (feat[c] % 1 == 0).all(), f'{c} có giá trị không nguyên'
assert (feat['BMI'] > 0).all(), 'BMI phải > 0'
print('Miền giá trị: đạt (14 biến nhị phân, 6 biến mã/số ngày, BMI > 0)')

Miền giá trị: đạt (14 biến nhị phân, 6 biến mã/số ngày, BMI > 0)


## 4. Vector đặc trưng trùng nhưng nhãn khác nhau

Con số này dùng để đối chiếu chéo với `dataset_audit.json` của A. Đây chỉ là thống kê, **không** dùng để xóa dòng hay sửa nhãn.

In [6]:
feature_cols = list(feat.columns)
labels_per_vector = df.groupby(feature_cols)[TARGET].nunique()
conflict_groups = int((labels_per_vector > 1).sum())
unique_vectors = int(len(labels_per_vector))

print(f'Số vector đặc trưng duy nhất               : {unique_vectors:,}')
print(f'Số vector trùng nhưng nhãn khác nhau (nhóm): {conflict_groups:,}')

Số vector đặc trưng duy nhất               : 227,908
Số vector trùng nhưng nhãn khác nhau (nhóm): 1,566


## 5. Kích thước chia tập kỳ vọng (70/15/15, seed 42)

A dùng `round(0.15·N)` cho val và test, phần còn lại là train. Đối chiếu với idx của B và `split_indices.npz` của A.

In [7]:
n = len(df)
n_val = round(0.15 * n)
n_test = round(0.15 * n)
n_train = n - n_val - n_test
print(f'Train/Val/Test kỳ vọng: {n_train:,} / {n_val:,} / {n_test:,}')
assert (n_train, n_val, n_test) == (177_576, 38_052, 38_052)

Train/Val/Test kỳ vọng: 177,576 / 38,052 / 38,052
